# A* Pathfinder 🧭
How do game characters and route planners find the shortest path so fast? **A\*** explores the most promising squares first, using a heuristic (an optimistic guess of the remaining distance). Implement it, race it against breadth-first search on a random maze, and watch how many fewer squares it needs to look at.

Background: [[RL Basics and MDPs]] (planning) · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import heapq, collections, numpy as np, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
rng = np.random.default_rng(4); N = 60
grid = rng.random((N, N)) < 0.28; grid[0, 0] = grid[N - 1, N - 1] = False          # True = wall
START, GOAL = (0, 0), (N - 1, N - 1)
def neighbours(p):
    r, c = p
    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        q = (r + dr, c + dc)
        if 0 <= q[0] < N and 0 <= q[1] < N and not grid[q]: yield q
def bfs(start, goal):
    parent = {start: None}; queue = collections.deque([start]); expanded = 0
    while queue:
        node = queue.popleft(); expanded += 1
        if node == goal:
            path = []
            while node is not None: path.append(node); node = parent[node]
            return path[::-1], expanded
        for nb in neighbours(node):
            if nb not in parent: parent[nb] = node; queue.append(nb)
    return None, expanded
for seed in range(4, 100):                                      # draw mazes until one has a path from corner to corner
    grid = np.random.default_rng(seed).random((N, N)) < 0.28; grid[0, 0] = grid[N - 1, N - 1] = False
    if bfs(START, GOAL)[0]: break
print(f"maze seed {seed}: {grid.mean():.0%} walls")

## 1. The heuristic
On a 4-connected grid with unit steps the **Manhattan distance** never overestimates the true distance (it's *admissible*), so A* stays optimal.

In [ ]:
def manhattan(a, b):
    # TODO 1: |Δrow| + |Δcol|
    raise NotImplementedError  # TODO 1

## 2. A*
Keep a priority queue ordered by $f = g + h$ ($g$ = cost so far). Pop the lowest $f$; skip stale entries; stop when you pop the goal; otherwise relax each neighbour (cost +1). Return `(path, number_of_expanded_nodes)`; `path` runs from start to goal inclusive, or `None`.

In [ ]:
def astar(start, goal, h):
    # TODO 2: A* with heapq; count a node as expanded when you pop it (non-stale)
    raise NotImplementedError  # TODO 2

In [ ]:
check("heuristic", lambda: manhattan((0, 0), (3, 4)) == 7 and manhattan((5, 1), (2, 1)) == 3, "|3| + |4| = 7.")
if ready("heuristic"):
    try:
        p_bfs, e_bfs = bfs(START, GOAL); p_dij, e_dij = astar(START, GOAL, lambda a, b: 0); p_a, e_a = astar(START, GOAL, manhattan)
        print(f"path length: BFS {len(p_bfs) - 1}, Dijkstra (h=0) {len(p_dij) - 1}, A* {len(p_a) - 1}")
        print(f"squares expanded: BFS {e_bfs}, Dijkstra {e_dij}, A* {e_a}")
        img = np.where(grid, 0.0, 1.0); fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        for a, (p, t) in zip(ax, [(p_bfs, f"BFS ({e_bfs} expanded)"), (p_a, f"A* ({e_a} expanded)")]):
            a.imshow(img, cmap="gray"); a.plot([q[1] for q in p], [q[0] for q in p], "r-", lw=2); a.set_title(t); a.axis("off")
        plt.show()
        check("A* is optimal and smarter", lambda: len(p_a) == len(p_bfs) and e_a < 0.7 * e_bfs, "A* must find an equally short path while expanding < 70 % of BFS's squares.")
    except NotImplementedError:
        print("❌ A* is optimal and smarter: not implemented yet – fill in the TODO above.")

<details><summary>▶ Solution</summary>

```python
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])
```

```python
def astar(start, goal, h):
    open_heap = [(h(start, goal), 0, start)]
    g = {start: 0}; parent = {start: None}; expanded = 0
    while open_heap:
        f, cost, node = heapq.heappop(open_heap)
        if cost > g[node]:
            continue
        expanded += 1
        if node == goal:
            path = []
            while node is not None:
                path.append(node); node = parent[node]
            return path[::-1], expanded
        for nb in neighbours(node):
            new = cost + 1
            if new < g.get(nb, float("inf")):
                g[nb] = new; parent[nb] = node
                heapq.heappush(open_heap, (new + h(nb, goal), new, nb))
    return None, expanded
```
</details>

## Stretch goals
- An *inadmissible* heuristic: `2 * manhattan`. Faster? Still optimal?
- Weighted terrain (mud costs 5): Dijkstra and A* handle it, BFS doesn't.
- Real streets: download Aalborg's road graph with `osmnx` and route from AAU to the train station.